In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
normal_data = pd.read_csv('normal_data.csv')

In [3]:
normal_data.head()

,Unnamed: 0
0,"ax_g,ay_g,az_g,accel_magnitude"
1,"-0.1165,0.8811,-0.4412,0.9922"
2,"-0.1162,0.8723,-0.4470,0.9870"
3,"-0.1155,0.8723,-0.4341,0.9812"
4,"-0.1182,0.8760,-0.4436,0.9890"


In [4]:
disturbed_data = pd.read_csv('disturbed_data.csv')

In [5]:
disturbed_data.head()

,Unnamed: 0
0,"ax_g,ay_g,az_g,accel_magnitude"
1,"-0.1299,0.8684,-0.4441,0.9840"
2,"-0.1292,0.8755,-0.4504,0.9930"
3,"-0.1311,0.8743,-0.4426,0.9887"
4,"-0.1306,0.8743,-0.4438,0.9891"


In [6]:
normal_data = normal_data.iloc[:, 0].str.split(',', expand=True)
disturbed_data = disturbed_data.iloc[:, 0].str.split(',', expand=True)


In [7]:
normal_data.shape

(2051, 4)

In [8]:
normal_data.head()

,0,1,2,3
0,ax_g,ay_g,az_g,accel_magnitude
1,-0.1165,0.8811,-0.4412,0.9922
2,-0.1162,0.8723,-0.4470,0.9870
3,-0.1155,0.8723,-0.4341,0.9812
4,-0.1182,0.8760,-0.4436,0.9890


In [9]:
disturbed_data.shape

(2192, 4)

In [10]:
disturbed_data.head()

,0,1,2,3
0,ax_g,ay_g,az_g,accel_magnitude
1,-0.1299,0.8684,-0.4441,0.9840
2,-0.1292,0.8755,-0.4504,0.9930
3,-0.1311,0.8743,-0.4426,0.9887
4,-0.1306,0.8743,-0.4438,0.9891


In [11]:
normal_data.to_csv('normal_data_clean.csv',index = False)
disturbed_data.to_csv('disturbed_data_clean.csv',index = False)

In [12]:
print(normal_data.columns)

RangeIndex(start=0, stop=4, step=1)


In [13]:
normal_data.columns = ['ax_g', 'ay_g', 'az_g', 'accel_magnitude']
disturbed_data.columns = ['ax_g', 'ay_g', 'az_g', 'accel_magnitude']


In [14]:
print(normal_data.columns)

Index(['ax_g', 'ay_g', 'az_g', 'accel_magnitude'], dtype='object')


In [15]:
# Cleaning the data
normal_data['accel_magnitude'] = normal_data['accel_magnitude'].str.strip()
disturbed_data['accel_magnitude'] = disturbed_data['accel_magnitude'].str.strip()
# removes junk,spaces,etc

In [16]:
# making junk data to NaN
normal_data['accel_magnitude'] = pd.to_numeric(
    normal_data['accel_magnitude'],
    errors = 'coerce'
)
disturbed_data['accel_magnitude'] = pd.to_numeric(
    disturbed_data['accel_magnitude'],
    errors = 'coerce'
)

In [17]:
# drop bad rows
normal_data = normal_data.dropna(subset=['accel_magnitude'])
disturbed_data = disturbed_data.dropna(subset=['accel_magnitude'])

In [18]:
normal_mag = normal_data['accel_magnitude'].astype(float).values
normal_mag

array([0.9922, 0.987 , 0.9812, ..., 0.992 , 0.9965, 1.0006])

In [19]:
disturbed_mag = disturbed_data['accel_magnitude'].values
disturbed_mag

array([0.984 , 0.993 , 0.9887, ..., 0.989 , 0.9925, 0.9956])

In [20]:
type(normal_mag[0])

numpy.float64

In [21]:
# function to extract features

In [22]:
def extract_features(signal):
    return{
        'mean': np.mean(signal),
        'std': np.std(signal),
        'var' : np.var(signal),
        'rms' : np.sqrt(np.mean(signal**2)),
        'max' : np.max(signal),
        'min' : np.min(signal)
    
    }

In [23]:
normal_features = extract_features(normal_mag)
disturbed_features = extract_features(disturbed_mag)
normal_features , disturbed_features

({'mean': 0.9908324390243902,
  'std': 0.003852819860884793,
  'var': 1.4844220880428318e-05,
  'rms': 0.9908399297787219,
  'max': 1.0065,
  'min': 0.9783},
 {'mean': 1.0314934276586034,
  'std': 0.3277353716935846,
  'var': 0.10741047385913202,
  'rms': 1.0823073339685112,
  'max': 3.464,
  'min': 0.1524})

In [24]:
# creating feature table
features_df = pd.DataFrame([
    {**normal_features,'label':0},
    {**disturbed_features,'label':1}
])
features_df

,mean,std,var,rms,max,min,label
0,0.990832,0.003853,0.000015,0.990840,1.0065,0.9783,0
1,1.031493,0.327735,0.107410,1.082307,3.4640,0.1524,1


In [25]:
features_df.to_csv('day3_features.csv',index=False)

In [26]:
df = pd.read_csv("day3_features.csv")
df

,mean,std,var,rms,max,min,label
0,0.990832,0.003853,0.000015,0.990840,1.0065,0.9783,0
1,1.031493,0.327735,0.107410,1.082307,3.4640,0.1524,1


In [27]:
X = df.drop('label',axis=1)
y = df['label']
X,y

(       mean       std       var       rms     max     min
 0  0.990832  0.003853  0.000015  0.990840  1.0065  0.9783
 1  1.031493  0.327735  0.107410  1.082307  3.4640  0.1524,
 0    0
 1    1
 Name: label, dtype: int64)

In [28]:
# loaded the data and seperated input features (X) and output labels (y) for supervised learning

In [29]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,
                                                 test_size = 0.2,
                                                 random_state = 42,
                                                stratify = y)

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [30]:
# to get more datasets we'll be using windowing

In [34]:
normal_df = pd.read_csv('normal_data_clean.csv')
normal_df


,0,1,2,3
0,ax_g,ay_g,az_g,accel_magnitude
1,-0.1165,0.8811,-0.4412,0.9922
2,-0.1162,0.8723,-0.4470,0.9870
3,-0.1155,0.8723,-0.4341,0.9812
4,-0.1182,0.8760,-0.4436,0.9890
...,...,...,...,...
2046,-0.1174,0.8735,-0.4485,0.9889
2047,-0.1128,0.8804,-0.4456,0.9931
2048,-0.1182,0.8801,-0.4421,0.9920
2049,-0.1169,0.8865,-0.4399,0.9965


In [35]:
disturbed_df = pd.read_csv('disturbed_data_clean.csv')
disturbed_df

,0,1,2,3
0,ax_g,ay_g,az_g,accel_magnitude
1,-0.1299,0.8684,-0.4441,0.9840
2,-0.1292,0.8755,-0.4504,0.9930
3,-0.1311,0.8743,-0.4426,0.9887
4,-0.1306,0.8743,-0.4438,0.9891
...,...,...,...,...
2187,-0.1116,0.9016,-0.3926,0.9897
2188,-0.1143,0.8958,-0.3813,0.9802
2189,-0.1165,0.8992,-0.3950,0.9890
2190,-0.1169,0.9004,-0.4009,0.9925


In [37]:
normal_df = pd.read_csv('normal_data_clean.csv')
normal_signal = normal_df['accel_magnitude'].values

KeyError: 'accel_magnitude'

In [40]:
normal_df.columns

Index(['0', '1', '2', '3'], dtype='object')